In [16]:
import pandas as pd
from pathlib import Path


In [17]:
def main():
    # ---------- CONFIG: paths ----------
    base_dir = Path(".")  # change if needed

    daily_stats_path = base_dir / "stations_daily_statistics_mainland.csv"
    knn_weights_path = base_dir / "region_station_KNN_weights_mainland.csv"

    region_avg_out = base_dir / "region_avg.csv"
    region_avg_knn_out = base_dir / "region_avg_knn_weighted.csv"

    # ---------- LOAD DATA ----------
    daily_df = pd.read_csv(daily_stats_path, low_memory=False)
    weights_df = pd.read_csv(knn_weights_path, low_memory=False)

    # Standardize column names to lowercase for safety
    daily_df.columns = daily_df.columns.str.lower()
    weights_df.columns = weights_df.columns.str.lower()

    # Explicit column names (based on your files)
    date_col = "date"
    region_col = "region_code"
    station_col = "station_id"
    temp_col = "tmean_calc"

    weight_region_col = "region_code"
    weight_station_col = "station_id"
    weight_col = "weight"

    # ---------- 1) DAILY SIMPLE AVG TEMPERATURE PER REGION ----------
    # Mean of tmean_calc over stations within each region per day
    region_avg = (
        daily_df
        .groupby([date_col, region_col])[temp_col]
        .mean()
        .reset_index()
        .rename(columns={temp_col: "daily_avg_temperature"})
    )

    region_avg.to_csv(region_avg_out, index=False)
    print(f"Saved simple regional averages to: {region_avg_out}")

    # ---------- 2) DAILY KNN-WEIGHTED AVG TEMPERATURE PER REGION ----------
    # Merge station daily temps with region-station KNN weights
    merged_df = daily_df.merge(
        weights_df,
        left_on=[region_col, station_col],
        right_on=[weight_region_col, weight_station_col],
        how="inner",
        suffixes=("", "_w")
    )

    # Compute weighted temperature
    merged_df["weighted_temp"] = merged_df[temp_col] * merged_df[weight_col]

    # Sum weighted temps per date+region
    # (Assumes weights per region sum to 1; if not, you could divide by
    #  merged_df.groupby([date_col, region_col])[weight_col].sum() instead.)
    region_avg_knn = (
        merged_df
        .groupby([date_col, region_col])["weighted_temp"]
        .sum()
        .reset_index()
        .rename(columns={"weighted_temp": "daily_avg_temperature_knn_weighted"})
    )

    region_avg_knn.to_csv(region_avg_knn_out, index=False)
    print(f"Saved KNN-weighted regional averages to: {region_avg_knn_out}")


if __name__ == "__main__":
    main()

Saved simple regional averages to: region_avg.csv
Saved KNN-weighted regional averages to: region_avg_knn_weighted.csv
